# DVF: Predict Next Mutation
# Author: Xavier VAN AUSLOOS, 9/02/26
# Notebook for preparing dataset (train/test) and training ML models for predicting next mutation


In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


In [2]:
# Get project root and ensure directories exist
_project_root = Path.cwd() if (Path.cwd() / "src" / "dvf").exists() else Path.cwd().parent
data_dir = _project_root / "data" / "processed"
models_dir = _project_root / "data" / "models"
results_dir = _project_root / "data" / "results"

for d in [data_dir, models_dir, results_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {data_dir}")
print(f"Models dir: {models_dir}")
print(f"Results dir: {results_dir}")


Data dir: /Users/xaviervanausloos/projects/ldi_dvf/data/processed
Models dir: /Users/xaviervanausloos/projects/ldi_dvf/data/models
Results dir: /Users/xaviervanausloos/projects/ldi_dvf/data/results


In [3]:
# Load dataset from data/processed/
DATA_PATH = data_dir / "df_grouped_ensues_2020_2025_houses_cleaned.csv"
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded {len(df)} rows for houses mutations located in Ensues")
print(f"Columns: {list(df.columns)}")
df.head()


✅ Loaded 253 rows for houses mutations located in Ensues
Columns: ['Code postal', 'Commune', 'Code departement', 'Code commune', 'Prefixe de section', 'Section', 'No plan', 'Type local', 'Surface reelle bati', 'Surface terrain', 'Nombre pieces principales', 'mutations']


,Code postal,Commune,Code departement,Code commune,Prefixe de section,Section,No plan,Type local,Surface reelle bati,Surface terrain,Nombre pieces principales,mutations
0,13820.0,ENSUES-LA-REDONNE,13,33,NaN,AA,19,Maison,80.0,187.0,3.0,"[{'22/07/2021': '120000,00'}]"
1,13820.0,ENSUES-LA-REDONNE,13,33,NaN,AA,95,Maison,124.0,447.0,5.0,"[{'07/12/2022': '625000,00'}]"
2,13820.0,ENSUES-LA-REDONNE,13,33,NaN,AA,109,Maison,79.0,171.0,4.0,"[{'18/07/2022': '435550,00'}]"
3,13820.0,ENSUES-LA-REDONNE,13,33,NaN,AA,111,Maison,82.0,196.0,4.0,"[{'25/09/2023': '429500,00'}]"
4,13820.0,ENSUES-LA-REDONNE,13,33,NaN,AA,198,Maison,105.0,337.0,5.0,"[{'20/10/2022': '517000,00'}]"


In [4]:
# Parse mutations JSON and extract features for next mutation prediction
import ast

def parse_mutations(mutations_str):
    """Parse mutations JSON string (or Python dict string) and return list of (date, value) tuples.
    
    Handles both JSON format (double quotes) and Python dict format (single quotes).
    """
    if pd.isna(mutations_str) or not mutations_str:
        return []
    
    try:
        # Try ast.literal_eval first (handles Python-style single quotes)
        mutations = ast.literal_eval(mutations_str)
    except (ValueError, SyntaxError):
        try:
            # Fallback to json.loads (handles JSON double quotes)
            mutations = json.loads(mutations_str)
        except (json.JSONDecodeError, ValueError):
            return []
    
    parsed = []
    for mut in mutations:
        for date_str, value_str in mut.items():
            # Parse date
            date = pd.to_datetime(date_str, format="mixed", dayfirst=True, errors="coerce")
            # Parse value (handle comma decimal)
            value = pd.to_numeric(value_str.replace(",", "."), errors="coerce")
            if pd.notna(date) and pd.notna(value):
                parsed.append((date, value))
    return sorted(parsed, key=lambda x: x[0])  # Sort by date

# Extract mutation features
df["mutations_parsed"] = df["mutations"].apply(parse_mutations)
df["n_mutations"] = df["mutations_parsed"].apply(len)

# Filter: need at least 2 mutations to predict next one
df_predictable = df[df["n_mutations"] >= 2].copy()
print(f"✅ Houses with >= 2 mutations: {len(df_predictable)} / {len(df)}")

# check results for a specific house
# df_predictable[(df_predictable["Section"] == "AT") & (df_predictable["No plan"].astype(str) == "113")]


✅ Houses with >= 2 mutations: 16 / 253


In [5]:
df_predictable.head()

,Code postal,Commune,Code departement,Code commune,Prefixe de section,Section,No plan,Type local,Surface reelle bati,Surface terrain,Nombre pieces principales,mutations,mutations_parsed,n_mutations
5,13820.0,ENSUES-LA-REDONNE,13,33,NaN,AA,367,Maison,10.0,NaN,1.0,"[{'30/12/2020': '200000,00'}, {'05/05/2022': '...","[(2020-12-30 00:00:00, 200000.0), (2022-05-05 ...",2
12,13820.0,ENSUES-LA-REDONNE,13,33,NaN,AD,48,Maison,72.0,500.0,3.0,"[{'18/12/2020': '585000,00'}, {'24/08/2022': '...","[(2020-12-18 00:00:00, 585000.0), (2022-08-24 ...",2
26,13820.0,ENSUES-LA-REDONNE,13,33,NaN,AD,274,Maison,72.0,453.0,3.0,"[{'06/07/2022': '380000,00'}, {'05/12/2023': '...","[(2022-07-06 00:00:00, 380000.0), (2023-12-05 ...",2
46,13820.0,ENSUES-LA-REDONNE,13,33,NaN,AE,240,Maison,130.0,NaN,4.0,"[{'19/05/2021': '350000,00'}, {'09/05/2022': '...","[(2021-05-19 00:00:00, 350000.0), (2022-05-09 ...",2
71,13820.0,ENSUES-LA-REDONNE,13,33,NaN,AE,646,Maison,90.0,24.0,4.0,"[{'23/10/2020': '266100,00'}, {'10/10/2022': '...","[(2020-10-23 00:00:00, 266100.0), (2022-10-10 ...",2


In [6]:
# Create features and target for next mutation prediction
def extract_features(row):
    """Extract features from house and mutation history."""
    muts = row["mutations_parsed"]
    if len(muts) < 2:
        return None
    
    # Historical features
    dates = [m[0] for m in muts]
    values = [m[1] for m in muts]
    
    # Last mutation (will be used as target)
    last_date = dates[-1]
    last_value = values[-1]
    
    # Previous mutations (for features)
    prev_dates = dates[:-1]
    prev_values = values[:-1]
    
    # Time features
    # days_since_first: total days from first mutation to last mutation
    # Example: 06/08/2021 to 28/04/2023 = 630 days
    days_since_first = (last_date - dates[0]).days if len(dates) > 1 else 0
    avg_days_between = np.mean([(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]) if len(dates) > 1 else 0
    
    # Value features
    avg_value = np.mean(prev_values) if prev_values else 0
    value_trend = (last_value - prev_values[0]) / prev_values[0] if len(prev_values) > 0 and prev_values[0] > 0 else 0
    
    # Debug: Print calculation for Section AT, No plan 113
    try:
        if str(row["Section"]) == "AT" and str(row["No plan"]) == "113":
            print(f"DEBUG AT 113:")
            print(f"  dates[0] = {dates[0]} (type: {type(dates[0])})")
            print(f"  last_date = {last_date} (type: {type(last_date)})")
            print(f"  diff = {last_date - dates[0]}")
            print(f"  days_since_first = {days_since_first}")
            print(f"  dates = {dates}")
    except Exception as e:
        pass
    
    return {
        "n_prev_mutations": len(prev_values),
        "days_since_first": days_since_first,
        "avg_days_between": avg_days_between,
        "avg_prev_value": avg_value,
        "value_trend": value_trend,
        "last_value": last_value,
        "target_date": last_date,  # Next mutation date (using last as proxy)
        "target_value": last_value,  # Next mutation value
    }

# Extract features
features_list = []
for idx, row in df_predictable.iterrows():
    feat = extract_features(row)
    if feat:
        feat["idx"] = idx
        features_list.append(feat)

print(f"✅ Extracted features for {len(features_list)} houses")


DEBUG AT 113:
  dates[0] = 2021-08-06 00:00:00 (type: <class 'pandas.Timestamp'>)
  last_date = 2023-04-28 00:00:00 (type: <class 'pandas.Timestamp'>)
  diff = 630 days 00:00:00
  days_since_first = 630
  dates = [Timestamp('2021-08-06 00:00:00'), Timestamp('2023-04-28 00:00:00')]
✅ Extracted features for 16 houses


In [7]:
df_features = pd.DataFrame(features_list)

# Debug: Check days_since_first before merge (match on original index, do NOT reset_index)
if len(df_features) > 0:
    for feat in features_list:
        orig_idx = feat["idx"]
        if orig_idx in df_predictable.index:
            row = df_predictable.loc[orig_idx]
            if str(row["Section"]) == "AT" and str(row["No plan"]) == "113":
                print(f"DEBUG BEFORE MERGE (from features_list) - days_since_first: {feat['days_since_first']}")
                # Also check in df_features DataFrame
                feat_in_df = df_features[df_features["idx"] == orig_idx]
                if len(feat_in_df) > 0:
                    print(f"DEBUG BEFORE MERGE (from df_features DataFrame) - days_since_first: {feat_in_df.iloc[0]['days_since_first']}")
                break

# Merge: use original index so left_on="idx" matches right_index (no reset_index)
# Explicitly exclude calculated feature columns to prevent overwriting
calculated_feature_cols = ["n_prev_mutations", "days_since_first", "avg_days_between", 
                           "avg_prev_value", "value_trend", "last_value", "target_date", "target_value"]
cols_to_merge = [col for col in df_predictable.columns 
                 if col not in df_features.columns and col not in calculated_feature_cols]

if cols_to_merge:
    df_predictable_for_merge = df_predictable[cols_to_merge]
    df_features = df_features.merge(df_predictable_for_merge, left_on="idx", right_index=True, how="left")
else:
    # No columns to merge, just ensure idx alignment
    df_features = df_features.merge(df_predictable[[]], left_on="idx", right_index=True, how="left")

# Debug: Check days_since_first after merge
if len(df_features) > 0:
    mask_at113 = (df_features["Section"] == "AT") & (df_features["No plan"].astype(str) == "113")
    if mask_at113.any():
        row_after = df_features[mask_at113].iloc[0]
        print(f"DEBUG AFTER MERGE - days_since_first: {row_after['days_since_first']}")

print(f"Features extracted: {len(df_features)} rows")
df_features.head()


DEBUG BEFORE MERGE (from features_list) - days_since_first: 630
DEBUG BEFORE MERGE (from df_features DataFrame) - days_since_first: 630
DEBUG AFTER MERGE - days_since_first: 630
Features extracted: 16 rows


,n_prev_mutations,days_since_first,avg_days_between,avg_prev_value,value_trend,last_value,target_date,target_value,idx,Code postal,...,Prefixe de section,Section,No plan,Type local,Surface reelle bati,Surface terrain,Nombre pieces principales,mutations,mutations_parsed,n_mutations
0,1,491,491.0,200000.0,0.575000,315000.0,2022-05-05,315000.0,5,13820.0,...,NaN,AA,367,Maison,10.0,NaN,1.0,"[{'30/12/2020': '200000,00'}, {'05/05/2022': '...","[(2020-12-30 00:00:00, 200000.0), (2022-05-05 ...",2
1,1,614,614.0,585000.0,0.071624,626900.0,2022-08-24,626900.0,12,13820.0,...,NaN,AD,48,Maison,72.0,500.0,3.0,"[{'18/12/2020': '585000,00'}, {'24/08/2022': '...","[(2020-12-18 00:00:00, 585000.0), (2022-08-24 ...",2
2,1,517,517.0,380000.0,0.278947,486000.0,2023-12-05,486000.0,26,13820.0,...,NaN,AD,274,Maison,72.0,453.0,3.0,"[{'06/07/2022': '380000,00'}, {'05/12/2023': '...","[(2022-07-06 00:00:00, 380000.0), (2023-12-05 ...",2
3,1,355,355.0,350000.0,-0.338786,231425.0,2022-05-09,231425.0,46,13820.0,...,NaN,AE,240,Maison,130.0,NaN,4.0,"[{'19/05/2021': '350000,00'}, {'09/05/2022': '...","[(2021-05-19 00:00:00, 350000.0), (2022-05-09 ...",2
4,1,717,717.0,266100.0,0.379181,367000.0,2022-10-10,367000.0,71,13820.0,...,NaN,AE,646,Maison,90.0,24.0,4.0,"[{'23/10/2020': '266100,00'}, {'10/10/2022': '...","[(2020-10-23 00:00:00, 266100.0), (2022-10-10 ...",2


In [8]:
# Debug: print actual dates used for days_since_first (Section AT, No plan 113)
mask = (df_predictable["Section"] == "AT") & (df_predictable["No plan"].astype(str) == "113")
if mask.any():
    row = df_predictable.loc[mask].iloc[0]
    muts = row["mutations_parsed"]
    dates = [m[0] for m in muts]
    print("Raw mutations string (first 120 chars):", repr(row["mutations"])[:120], "...")
    print("Parsed dates:")
    for i, d in enumerate(dates):
        print(f"  {i}: {d} ({d.strftime('%d/%m/%Y') if hasattr(d, 'strftime') else d})")
    if len(dates) >= 2:
        first, last = dates[0], dates[-1]
        days = (last - first).days
        print(f"\nFirst: {first}, Last: {last}")
        print(f"days_since_first = (last - first).days = {days}")
else:
    print("No row with Section AT and No plan 113 found in df_predictable.")
    print("Showing first row with 2+ mutations:")
    row = df_predictable.iloc[0]
    muts = row["mutations_parsed"]
    dates = [m[0] for m in muts]
    print("Parsed dates:", dates)
    if len(dates) >= 2:
        print(f"days_since_first = {(dates[-1] - dates[0]).days}")

Raw mutations string (first 120 chars): "[{'06/08/2021': '642300,00'}, {'28/04/2023': '699900,00'}]" ...
Parsed dates:
  0: 2021-08-06 00:00:00 (06/08/2021)
  1: 2023-04-28 00:00:00 (28/04/2023)

First: 2021-08-06 00:00:00, Last: 2023-04-28 00:00:00
days_since_first = (last - first).days = 630


In [9]:
# Prepare features (X) and targets (y)
# Features: house characteristics + mutation history features
feature_cols = [
    "Code postal",
    "Code commune",
    "Surface reelle bati",
    "Surface terrain",
    "Nombre pieces principales",
    "n_prev_mutations",
    "days_since_first",
    "avg_days_between",
    "avg_prev_value",
    "value_trend",
]

# Targets: next mutation date (days from reference) and value
X = df_features[feature_cols].copy()
X = X.fillna(0)  # Fill NaN

# Target 1: Days until next mutation (relative to last known date)
reference_date = df_features["target_date"].max()
y_days = (df_features["target_date"] - reference_date).dt.days

# Target 2: Next mutation value
y_value = df_features["target_value"]

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y_days.shape}, {y_value.shape}")


Features shape: (16, 10)
Targets shape: (16,), (16,)


In [10]:
# Split into train/test sets
X_train, X_test, y_days_train, y_days_test, y_value_train, y_value_test = train_test_split(
    X, y_days, y_value, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train)} rows")
print(f"Test: {len(X_test)} rows")

# Save train/test sets
X_train.to_csv(data_dir / "X_train.csv", index=False)
X_test.to_csv(data_dir / "X_test.csv", index=False)
y_days_train.to_csv(data_dir / "y_days_train.csv", index=False)
y_days_test.to_csv(data_dir / "y_days_test.csv", index=False)
y_value_train.to_csv(data_dir / "y_value_train.csv", index=False)
y_value_test.to_csv(data_dir / "y_value_test.csv", index=False)

print(f"\nSaved train/test sets to {data_dir}")


Train: 12 rows
Test: 4 rows

Saved train/test sets to /Users/xaviervanausloos/projects/ldi_dvf/data/processed


In [11]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model 1: Predict days until next mutation
model_days = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_days.fit(X_train_scaled, y_days_train)

# Train model 2: Predict next mutation value
model_value = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_value.fit(X_train_scaled, y_value_train)

print("Models trained successfully")


Models trained successfully


In [12]:
# Save ML models
joblib.dump(model_days, models_dir / "model_predict_days.pkl")
joblib.dump(model_value, models_dir / "model_predict_value.pkl")
joblib.dump(scaler, models_dir / "scaler.pkl")

print(f"Saved models to {models_dir}")


Saved models to /Users/xaviervanausloos/projects/ldi_dvf/data/models


In [13]:
# Test ML models
# Predictions
y_days_pred = model_days.predict(X_test_scaled)
y_value_pred = model_value.predict(X_test_scaled)

# Metrics for days prediction
mae_days = mean_absolute_error(y_days_test, y_days_pred)
rmse_days = np.sqrt(mean_squared_error(y_days_test, y_days_pred))
r2_days = r2_score(y_days_test, y_days_pred)

# Metrics for value prediction
mae_value = mean_absolute_error(y_value_test, y_value_pred)
rmse_value = np.sqrt(mean_squared_error(y_value_test, y_value_pred))
r2_value = r2_score(y_value_test, y_value_pred)

print("=== Days Prediction Metrics ===")
print(f"MAE: {mae_days:.2f} days")
print(f"RMSE: {rmse_days:.2f} days")
print(f"R²: {r2_days:.4f}")

print("\n=== Value Prediction Metrics ===")
print(f"MAE: {mae_value:.2f} €")
print(f"RMSE: {rmse_value:.2f} €")
print(f"R²: {r2_value:.4f}")


=== Days Prediction Metrics ===
MAE: 359.90 days
RMSE: 408.80 days
R²: -0.9075

=== Value Prediction Metrics ===
MAE: 105179.20 €
RMSE: 140152.56 €
R²: -0.2680


In [14]:
# Save test results
results = {
    "model_days": {
        "mae": float(mae_days),
        "rmse": float(rmse_days),
        "r2": float(r2_days),
    },
    "model_value": {
        "mae": float(mae_value),
        "rmse": float(rmse_value),
        "r2": float(r2_value),
    },
    "n_train": len(X_train),
    "n_test": len(X_test),
}

import json
with open(results_dir / "test_results.json", "w") as f:
    json.dump(results, f, indent=2)

# Save predictions
predictions_df = pd.DataFrame({
    "y_days_test": y_days_test.values,
    "y_days_pred": y_days_pred,
    "y_value_test": y_value_test.values,
    "y_value_pred": y_value_pred,
})
predictions_df.to_csv(results_dir / "predictions.csv", index=False)

print(f"\nSaved results to {results_dir}")



Saved results to /Users/xaviervanausloos/projects/ldi_dvf/data/results
